In [1]:
# prep
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
len(documents)

72

In [5]:
documents[1]

{'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor this module, all you need is Python with Jupyter.\n\n## Prerequisites\n\nYou need the following:\n\n- Python (3.14 or later)\n- An [OpenAI account](https://openai.com/) (or an OpenAI-compatible\n  provider like Groq, Gemini, or Ollama)\n- Basic familiarity with Python and the command line\n\n## Creating the project\n\nWe\'ll start from scratch - no cloning needed. You\'ll create the\nproject yourself, step by step.\n\nFirst, install uv. It\'s a Python package manager, and I switched all my\nprojects to it because it\'s fast and convenient. Once I started using\nit, I never wanted to go back.\n\nOn Mac or Linux:\n\n```bash\ncurl -LsSf https://astral.sh/uv/install.sh | sh\n```\n\nOn Windows:\n\n```powershell\npowershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"\n```\n\n(You can also use `pip install uv` if you p

In [8]:
documents[1].keys()

dict_keys(['content', 'filename'])

In [6]:
# index with minsearch
from minsearch import Index

index=Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

# fit the docs to get index
index.fit(documents)

In [9]:
# search a query
query='How does the agentic loop keep calling the model until it stops?'

search_results=index.search(
    query,
    num_results=5
)

In [12]:
search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [14]:
search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [15]:
search_results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [19]:
def build_context(search_results):
        lines=[]
        for doc in search_results:
            lines.append(doc['content'])
            lines.append('filename : '+doc['filename'])
            lines.append('')
        return '\n'.join(lines).strip()

In [20]:
context=build_context(search_results)
context

'# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry out the task

In [32]:
import ollama
from ollama import chat

In [33]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

In [39]:
# RAG
class RAGBase:
    def __init__(self,index,instructions=INSTRUCTIONS,prompt_template=PROMPT_TEMPLATE,model='qwen3:0.6b'):
        self.index=index # is any param with a search method
        # self.llm=llm
        self.instructions=instructions
        # self.course=course
        self.prompt_template=prompt_template
        self.model=model

    # serach method
    def search(self,question,num_results=5):
        return self.index.search(
            question,
            num_results=num_results
        )

    # build context to get the context as a string to pass to llm as context in prompt
    def build_context(self,search_results):
        lines=[]
        for doc in search_results:
            lines.append(doc['content'])
            lines.append('filename : '+doc['filename'])
            lines.append('')
        return '\n'.join(lines).strip()

    # function to build prompt
    def build_prompt(self,question,search_results):
        # get context first
        context=build_context(search_results)
        prompt=USER_PROMPT_TEMPLATE.format(
            question=question,
            context=context
        )
        return prompt.strip()

    # llm method for generation where the prompt is sent
    def llm_with_history(self,prompt):
        messages=[
        {'role':'system','content':INSTRUCTIONS},
        {'role':'user','content':prompt}
        ]
        response:ChatResponse=chat(
            model=self.model,
            messages=messages
        )
        return response

    # rag method which puts all functions together
    def rag(self,query):
        search_results=self.search(query)
        prompt=self.build_prompt(query,search_results)
        answer=self.llm_with_history(prompt)
        return answer


In [40]:
Rag=RAGBase(index=index)

In [41]:
ans=Rag.rag(query='How does the agentic loop keep calling the model until it stops?')
ans

ChatResponse(model='qwen3:0.6b', created_at='2026-06-22T17:06:29.1355023Z', done=True, done_reason='stop', total_duration=231475275600, load_duration=698645800, prompt_eval_count=4095, prompt_eval_duration=39795574000, eval_count=1301, eval_duration=190645695000, message=Message(role='assistant', content='To implement an **agent loop** with function calls, you need to structure your system to:\n\n1. **Send messages to the LLM** it handles.\n2. **Run function calls** (tools) it uses.\n3. **Add new messages** to the messages list as the model processes them.\n4. **Repeat** until the model is satisfied.\n\nHere\'s how you can implement this:\n\n---\n\n### **Example Implementation**\n\n```python\nmessages = []\n\ndef run_agent_loop():\n    # Step 1: Send messages to LLM\n    messages.append("This is your first message.")\n    \n    # Step 2: Run a function call (tool)\n    # This is a placeholder for a real LLM function call\n    function_result = process_function_call()\n    \n    # Step 

In [43]:
print(f'The number of input tokens is {ans.prompt_eval_count}')

The number of input tokens is 4095


In [44]:
prompt_tokens = ans.prompt_eval_count
completion_tokens = ans.eval_count
total_tokens = prompt_tokens + completion_tokens

print(f"Prompt tokens: {prompt_tokens}")
print(f"Completion tokens: {completion_tokens}")
print(f"Total tokens: {total_tokens}")

Prompt tokens: 4095
Completion tokens: 1301
Total tokens: 5396


In [37]:
# chunking

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [38]:
len(chunks)

295

In [46]:
# rag using chunks as doc

chunk_index=Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

# get index by fitting on chunks docs
chunk_index.fit(chunks)

In [47]:
Rag_using_chunks=RAGBase(index=chunk_index)

In [49]:
ans_chunk=Rag_using_chunks.rag(query='How does the agentic loop keep calling the model until it stops?')
ans_chunk

ChatResponse(model='qwen3:0.6b', created_at='2026-06-22T17:11:25.2533506Z', done=True, done_reason='stop', total_duration=68494247700, load_duration=690591400, prompt_eval_count=2340, prompt_eval_duration=17294483000, eval_count=393, eval_duration=50234149000, message=Message(role='assistant', content="The agentic loop keeps calling the model until it returns a response without function calls. Here's how it works:\n\n1. **Iteration Loop**: The loop keeps running until the model returns a final answer with no more tool calls.\n2. **Check for Function Calls**: During each iteration, the code checks if there are function calls in the response.\n3. **Process Output**: If a function call is found, it prints the function details and executes the tool, appending the result to the messages.\n4. **Termination**: The loop stops when there are no more function calls, meaning the model has provided the final answer.\n\nThe iteration counter tracks the number of round-trips, and the loop stops when

In [50]:
print(f'The number of input tokens is {ans_chunk.prompt_eval_count}')

The number of input tokens is 2340


In [51]:
prompt_tokens = ans_chunk.prompt_eval_count
completion_tokens = ans_chunk.eval_count
total_tokens = prompt_tokens + completion_tokens

print(f"Prompt tokens: {prompt_tokens}")
print(f"Completion tokens: {completion_tokens}")
print(f"Total tokens: {total_tokens}")

Prompt tokens: 2340
Completion tokens: 393
Total tokens: 2733


In [52]:
# turning into agent
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [54]:
def search(query: str) -> dict[str, str]:
    """
    Search the course database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5
    )

In [56]:
# register tool
agent_tools = Tools()
agent_tools.add_tool(search)

In [57]:
# lllok at tools
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [61]:
# create chat interface and runner 
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
    model="qwen3:0.6b",
    api_key="ollama",
    base_url="http://localhost:11434/v1")
)

TypeError: OpenAIClient.__init__() got an unexpected keyword argument 'api_key'

In [62]:
import inspect

print(inspect.signature(OpenAIClient))

(model: str = 'gpt-4o-mini', client: openai.OpenAI = None, extra_kwargs: dict = None)


In [63]:
import inspect
from toyaikit.llm import OpenAIClient

print(inspect.signature(OpenAIClient))

(model: str = 'gpt-4o-mini', client: openai.OpenAI = None, extra_kwargs: dict = None)


In [64]:
print(inspect.getsource(OpenAIClient))

class OpenAIClient(LLMClient):
    def __init__(
        self,
        model: str = "gpt-4o-mini",
        client: OpenAI = None,
        extra_kwargs: dict = None,
    ):
        self.model = model

        if client is None:
            self.client = OpenAI()
        else:
            self.client = client

        self.extra_kwargs = extra_kwargs or {}

    def send_request(
        self,
        chat_messages: List,
        tools: Tools = None,
        output_format: BaseModel = None,
    ) -> Response | ParsedResponse:
        tools_list = []

        if tools is not None:
            tools_list = tools.get_tools()

        args = dict(
            model=self.model,
            input=chat_messages,
            tools=tools_list,
            **self.extra_kwargs,
        )

        if output_format is not None:
            return self.client.responses.parse(
                text_format=output_format,
                **args,
            )

        return self.client.responses.create(**

In [65]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

llm_client = OpenAIClient(
    model="qwen3:0.6b",
    client=ollama_client
)

In [70]:
instructions="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

In [71]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client,
)

In [72]:
# run
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received
